# 学生建模沙盒

这是一个**可独立运行、可编辑的合成教学练习**。十个实体、连续 `response` 与二元 `hit`
标签均为编造数据，不是分子、实验测定、QSAR 证据或生物学证据。

**问题。** 如何把拟合、基于验证集的选择，以及留出集误差分析连起来，同时不让测试标签
参与模型选择？需要基础 Python，以及训练/验证/测试划分的概念。

**上传 Colab。** 下载此文件，打开 [Google Colab](https://colab.research.google.com/)，选择
**File → Upload notebook** 后选中该文件。CPU 足够；无需安装软件包、API key、网络请求、GPU
或 Drive 挂载。请从上到下运行。报告写入当前运行时中的新临时文件夹；如需保留，请自行下载。

## 1. 导入与新输出文件夹

这里仅使用 Python 标准库，以及 Jupyter 和 Colab 自带的 `IPython.display`。`mkdtemp`
会在当前工作目录下创建新目录。本 notebook 不会覆盖之前的报告，也不会改动作者结果文件。

In [ ]:
import json
import math
import tempfile
from pathlib import Path

from IPython.display import HTML, SVG, display

OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="student-modelling-", dir=Path.cwd()))
print("Fresh student output:", OUTPUT_DIR)

## 2. 拟合前检查构造数据

每个 `entity_id` 都唯一。这个标量 `feature` 特意不是分子描述符。最后两个记录构成测试划分：
一个是外推，另一个与训练直线矛盾。只有在先预测哪些量应改变后，才修改一个**副本**。

In [ ]:
def teaching_rows():
    """Authored numbers, not molecules, assay measurements, or a benchmark."""
    values = [
        (0, 1, 0, "train"),
        (1, 3, 0, "train"),
        (2, 5, 0, "train"),
        (3, 7, 1, "train"),
        (4, 9, 1, "train"),
        (5, 11, 1, "train"),
        (1.5, 4, 0, "validation"),
        (3.5, 8, 1, "validation"),
        (6, 10, 1, "test"),
        (7, 9, 0, "test"),
    ]
    return [
        {"entity_id": f"toy-{i + 1:02}", "feature": x, "response": y, "hit": hit, "split": split}
        for i, (x, y, hit, split) in enumerate(values)
    ]

rows = teaching_rows()
rows

## 3. 精确的拟合与评分机制

先阅读这些简短纯函数，再看 protocol cell。Ridge 最小化残差平方和加 `alpha * slope²`，截距
不受惩罚。Logistic 在仅用训练集标准化后，对平均交叉熵加 `alpha * slope² / 2` 做 500 次固定
全批量更新。概率仅在计算 log loss 时被截断。这些是教学机制，不是测定模型，也不保证收敛。

In [ ]:
def mean(values):
    return sum(values) / len(values)

def fit_ridge(x, y, alpha):
    """Minimize sum squared error + alpha * slope squared; intercept is free."""
    x_bar, y_bar = mean(x), mean(y)
    denominator = sum((value - x_bar) ** 2 for value in x) + alpha
    slope = (
        sum((a - x_bar) * (b - y_bar) for a, b in zip(x, y, strict=True)) / denominator
        if denominator
        else 0.0
    )
    return y_bar - slope * x_bar, slope

def sigmoid(value):
    if value >= 0:
        return 1 / (1 + math.exp(-value))
    exponential = math.exp(value)
    return exponential / (1 + exponential)

def fit_logistic(x, y, alpha, steps=500, start=(0.0, 0.0)):
    """Fixed full-batch steps on mean CE + alpha * slope squared / 2.

    The step budget and the starting point are arguments so that the optimiser
    itself can be varied without touching the data, the penalty or the selection
    rule. Defaults reproduce the original fit exactly.
    """
    center = mean(x)
    scale = math.sqrt(mean([(value - center) ** 2 for value in x])) or 1.0
    z = [(value - center) / scale for value in x]
    intercept, slope = start
    for _ in range(steps):
        residuals = [
            sigmoid(intercept + slope * value) - target for value, target in zip(z, y, strict=True)
        ]
        intercept -= 0.1 * mean(residuals)
        slope -= 0.1 * (mean([a * b for a, b in zip(residuals, z, strict=True)]) + alpha * slope)
    return {"intercept": intercept, "slope": slope, "mean": center, "scale": scale}

def predict_logistic(model, x):
    return [
        sigmoid(model["intercept"] + model["slope"] * (value - model["mean"]) / model["scale"])
        for value in x
    ]

def log_loss(y, probabilities):
    clipped = [min(max(value, 1e-15), 1 - 1e-15) for value in probabilities]
    return -mean(
        [
            target * math.log(p) + (1 - target) * math.log1p(-p)
            for target, p in zip(y, clipped, strict=True)
        ]
    )

def classification_metrics(y, probabilities):
    pairs = list(zip(y, [p >= 0.5 for p in probabilities], strict=True))
    tp = sum(target == 1 and prediction for target, prediction in pairs)
    tn = sum(target == 0 and not prediction for target, prediction in pairs)
    fp = sum(target == 0 and prediction for target, prediction in pairs)
    fn = sum(target == 1 and not prediction for target, prediction in pairs)
    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": (tp + tn) / len(y),
        "precision": tp / (tp + fp) if tp + fp else None,
        "recall": tp / (tp + fn) if tp + fn else None,
    }

def regression_metrics(y, predictions):
    errors = [prediction - target for target, prediction in zip(y, predictions, strict=True)]
    return {
        "mae": mean([abs(value) for value in errors]),
        "rmse": math.sqrt(mean([value**2 for value in errors])),
    }

def fit_stump(x, y, min_leaf):
    """Pick the training split that minimises squared error, both sides at least min_leaf.

    This is the smallest member of the tree family: one threshold, two constants.
    It shares nothing with the linear family except the records it is given, which
    is what makes the comparison a comparison of assumptions rather than of tuning.
    """
    order = sorted(zip(x, y, strict=True))
    thresholds = sorted({(a[0] + b[0]) / 2 for a, b in zip(order, order[1:], strict=False)})
    best = None
    for threshold in thresholds:
        left = [value for feature, value in order if feature < threshold]
        right = [value for feature, value in order if feature >= threshold]
        if len(left) < min_leaf or len(right) < min_leaf:
            continue
        left_mean, right_mean = mean(left), mean(right)
        error = sum((value - left_mean) ** 2 for value in left) + sum(
            (value - right_mean) ** 2 for value in right
        )
        if best is None or error < best[0]:
            best = (error, threshold, left_mean, right_mean)
    if best is None:
        # No admissible split: the family degenerates to the constant it contains.
        return None, mean(y), mean(y)
    return best[1], best[2], best[3]

def predict_stump(model, x):
    threshold, left, right = model
    if threshold is None:
        return [left] * len(x)
    return [left if value < threshold else right for value in x]

## 4. 拟合前验证与划分

这是规范 protocol 的第一个可见片段，而不是调用 runner。它在提取训练和验证值之前拒绝重复 ID、
无效划分和无效标签。此处不读取测试目标。

In [ ]:
"""Split first. Fit train only; select on validation; score test only afterward."""
if not rows or len({row["entity_id"] for row in rows}) != len(rows):
    raise ValueError("Entity IDs must be unique across all partitions")
partitions = {name: [] for name in ("train", "validation", "test")}
for row in rows:
    if row["split"] not in partitions:
        raise ValueError("Unknown partition")
    if not all(math.isfinite(row[key]) for key in ("feature", "response", "hit")):
        raise ValueError("Features and labels must be finite")
    if row["hit"] not in (0, 1):
        raise ValueError("The constructed hit target must be binary")
    partitions[row["split"]].append(row)
if any(not values for values in partitions.values()):
    raise ValueError("Each partition must contain records")
train, validation, test = (partitions[name] for name in ("train", "validation", "test"))
x = [row["feature"] for row in train]
y = [row["response"] for row in train]
hits = [row["hit"] for row in train]
validation_x = [row["feature"] for row in validation]

print({name: len(values) for name, values in partitions.items()})

## 5. 只用训练记录拟合候选模型

这里在**同一批训练记录**上拟合两个族。第一个是线性族：对响应用 ridge、对构造的 hit 用
logistic，各跑三个惩罚。第二个是树族中最小的成员——决策桩：选一个阈值，在两侧各预测一个常数，
跑三个最小叶大小。

惩罚与叶大小扮演同一个角色：都在约束该族可以多大程度地迁就训练记录。两个族真正不同的
不是调参，而是它们**对关系形状的假设**——这正是本节要暴露的选择。两个循环都不查看测试标签。

In [ ]:
candidates, validation_scores = {}, []
for alpha in (0.0, 1.0, 10.0):
    regression = fit_ridge(x, y, alpha)
    classification = fit_logistic(x, hits, alpha)
    candidates[str(alpha)] = {"regression": regression, "classification": classification}
    predictions = [regression[0] + regression[1] * value for value in validation_x]
    validation_scores.append(
        {
            "alpha": alpha,
            "regression_mae": regression_metrics(
                [row["response"] for row in validation],
                predictions,
            )["mae"],
            "classification_log_loss": log_loss(
                [row["hit"] for row in validation],
                predict_logistic(classification, validation_x),
            ),
        }
    )
stump_candidates, stump_validation_scores = {}, []
for min_leaf in (1, 2, 3):
    stump = fit_stump(x, y, min_leaf)
    stump_candidates[str(min_leaf)] = stump
    stump_validation_scores.append(
        {
            "min_leaf": min_leaf,
            "regression_mae": regression_metrics(
                [row["response"] for row in validation],
                predict_stump(stump, validation_x),
            )["mae"],
            "split_threshold": stump[0],
        }
    )

## 6. 先检查验证分数，再选择

先运行并检查这个 cell，再运行下一个。它为回归选择较低验证 MAE，为分类选择较低验证 log loss；
完全并列时选较小 alpha。**同一条规则**随后用于选择决策桩的叶大小，因此两个族在同一套程序下
被选出，差别只在于各自的假设。该表特意在任何测试评分之前展示。

继续之前请先读决策桩那张表。这里三个叶大小给出**相同的阈值与相同的验证 MAE**，
所以这个网格什么也没决定，是并列规则选走了约束最少的那个候选。
**一个分不开候选的网格，是关于这份数据的事实，不是把网格加宽到有人胜出的理由。**

In [ ]:
# Tuple ordering fixes ties in favor of the lower alpha; no test target is used here.
regression_alpha = min(
    validation_scores, key=lambda row: (row["regression_mae"], row["alpha"])
)["alpha"]
classification_alpha = min(
    validation_scores, key=lambda row: (row["classification_log_loss"], row["alpha"])
)["alpha"]
regression = candidates[str(regression_alpha)]["regression"]
classification = candidates[str(classification_alpha)]["classification"]
# The same rule, applied to the other family: lowest validation MAE, ties to the
# less constrained candidate. Only the assumption set differs.
stump_min_leaf = min(
    stump_validation_scores, key=lambda row: (row["regression_mae"], row["min_leaf"])
)["min_leaf"]
stump = stump_candidates[str(stump_min_leaf)]

display(HTML('<pre>' + json.dumps(validation_scores, indent=2) + '</pre>'))

## 7. 只测量优化过程本身贡献了什么

配置已经固定。本 cell 不改变数据、惩罚或选择规则，只改变搜索本身：三个写明的起始点
与三个步数预算的交叉。**过程中没有任何随机性**，因此下面的差异全部来自优化，而不是种子。

先读预算表。算力按**梯度评估次数**计（步数乘以六条训练记录），对每个预算问两个问题：
三个起始点在 0.5 阈值上是否给出相同的**决策**，以及它们的概率还相差多远。

起始点之间的跨度**不是置信区间**。它度量的是搜索走了多远，而不是这六条记录把答案约束得
多紧。一个收敛良好但数据严重不足的优化过程，跨度会很小，却依然不确定。

In [ ]:
# With the configuration chosen, vary only the optimiser. Three declared
# starting points and three step budgets, no randomness and no new data, so
# every difference below is the search and not the records.
optimisation_variability = []
for steps in (50, 200, 500):
    for start in ((0.0, 0.0), (-2.0, 1.0), (2.0, -1.0)):
        attempt = fit_logistic(x, hits, classification_alpha, steps=steps, start=start)
        validation_probabilities = predict_logistic(attempt, validation_x)
        optimisation_variability.append(
            {
                "steps": steps,
                "start": list(start),
                "gradient_evaluations": steps * len(train),
                "train_log_loss": log_loss(hits, predict_logistic(attempt, x)),
                "validation_probabilities": validation_probabilities,
                "validation_decisions": [
                    int(probability >= 0.5) for probability in validation_probabilities
                ],
            }
        )
budget_summary = []
for steps in (50, 200, 500):
    attempts = [row for row in optimisation_variability if row["steps"] == steps]
    decisions = {tuple(row["validation_decisions"]) for row in attempts}
    spreads = [
        max(column) - min(column)
        for column in zip(*(row["validation_probabilities"] for row in attempts), strict=True)
    ]
    budget_summary.append(
        {
            "steps": steps,
            "gradient_evaluations": attempts[0]["gradient_evaluations"],
            "starting_points": len(attempts),
            "decisions_agree": len(decisions) == 1,
            "largest_probability_spread": max(spreads),
        }
    )

display(HTML('<pre>' + json.dumps(budget_summary, indent=2) + '</pre>'))

## 8. 仅一次评分留出记录

仅在选择之后，这个最终规范片段才读取测试记录、产生预测并计算误差。它不使用验证行重新拟合。
请阅读报告，避免用这些值改动已经选择的 recipe。

In [ ]:
test_x = [row["feature"] for row in test]
predictions = [regression[0] + regression[1] * value for value in test_x]
stump_predictions = predict_stump(stump, test_x)
probabilities = predict_logistic(classification, test_x)
targets = [row["response"] for row in test]
test_hits = [row["hit"] for row in test]
student_report = {
    "data_kind": "synthetic-teaching-example",
    "claim": "Training mechanics only; not QSAR performance or biological evidence",
    "counts": {name: len(values) for name, values in partitions.items()},
    "selection": {
        "regression_alpha": regression_alpha,
        "stump_min_leaf": stump_min_leaf,
        "classification_alpha": classification_alpha,
        "classification_threshold": 0.5,
        "refit_with_validation": False,
    },
    "validation_scores": validation_scores,
    "stump_validation_scores": stump_validation_scores,
    "optimisation_variability": optimisation_variability,
    "budget_summary": budget_summary,
    "candidate_models": candidates,
    "models": {
        "regression": regression,
        "classification": classification,
        "stump": stump,
        "train_mean": mean(y),
        "train_prevalence": mean(hits),
    },
    "test_metrics": {
        "ridge": regression_metrics(targets, predictions),
        "stump": regression_metrics(targets, stump_predictions),
        "train_mean": regression_metrics(targets, [mean(y)] * len(test)),
        "logistic": {
            **classification_metrics(test_hits, probabilities),
            "log_loss": log_loss(test_hits, probabilities),
        },
        "train_prevalence": {
            **classification_metrics(test_hits, [mean(hits)] * len(test)),
            "log_loss": log_loss(test_hits, [mean(hits)] * len(test)),
        },
    },
    "test_predictions": [
        {
            **row,
            "prediction": prediction,
            "residual": prediction - row["response"],
            "hit_probability": probability,
        }
        for row, prediction, probability in zip(test, predictions, probabilities, strict=True)
    ],
}

display(HTML('<pre>' + json.dumps(student_report, indent=2) + '</pre>'))

## 9. 绘制实际拟合的结果

下方 SVG 直接由当前记录和所选模型生成。圆点按划分显示连续 response；深色直线是所选 ridge
预测。response 坐标轴包含整条已绘制外推线。柱形显示所选 logistic 概率与 0.5 参考线。
这只是一个小型可视化检查，不代表模型质量结论。

In [ ]:
width, height, left, bottom = 820, 430, 70, 330
features = [row["feature"] for row in rows]
responses = [row["response"] for row in rows]
x_min, x_max = min(features) - 0.5, max(features) + 0.5
intercept, slope = student_report["models"]["regression"]
displayed_predictions = [intercept + slope * value for value in (x_min, x_max)]
y_min = math.floor(min(*responses, *displayed_predictions) - 1)
y_max = math.ceil(max(*responses, *displayed_predictions) + 1)
def x_to_px(value):
    return left + (value - x_min) / (x_max - x_min) * 290

def y_to_px(value):
    return bottom - (value - y_min) / (y_max - y_min) * 250
colors = {"train": "#2563eb", "validation": "#d97706", "test": "#dc2626"}
line = " ".join(
    f"{x_to_px(value):.1f},{y_to_px(intercept + slope * value):.1f}"
    for value in (x_min, x_max)
)
circles = "".join(
    f'<circle cx="{x_to_px(row["feature"]):.1f}" cy="{y_to_px(row["response"]):.1f}" r="5" fill="{colors[row["split"]]}"/>'
    for row in rows
)
probabilities = predict_logistic(
    student_report["models"]["classification"], features
)
probability_left, probability_top, probability_bottom = 470, 80, bottom
def probability_to_px(value):
    return probability_bottom - value * (probability_bottom - probability_top)
bars = "".join(
    f'<rect x="{probability_left + 20 + index * 25}" y="{probability_to_px(probability):.1f}" width="15" height="{probability_bottom - probability_to_px(probability):.1f}" fill="#7c3aed"/>'
    for index, probability in enumerate(probabilities)
)
response_ticks = "".join(
    f'<line x1="{left - 4}" y1="{y_to_px(value):.1f}" x2="{left}" y2="{y_to_px(value):.1f}" stroke="black"/><text x="{left - 9}" y="{y_to_px(value) + 4:.1f}" text-anchor="end" font-size="11">{value}</text>'
    for value in range(y_min, y_max + 1, 2)
)
feature_ticks = "".join(
    f'<text x="{x_to_px(value):.1f}" y="{bottom + 18}" text-anchor="middle" font-size="11">{value}</text>'
    for value in range(math.ceil(x_min), math.floor(x_max) + 1)
)
probability_ticks = "".join(
    f'<text x="{probability_left - 9}" y="{probability_to_px(value) + 4:.1f}" text-anchor="end" font-size="11">{value:.1f}</text>'
    for value in (0.0, 0.5, 1.0)
)
entity_labels = "".join(
    f'<text x="{probability_left + 28 + index * 25}" y="{bottom + 18}" text-anchor="end" font-size="9" transform="rotate(-55 {probability_left + 28 + index * 25} {bottom + 18})">{row["entity_id"]}</text>'
    for index, row in enumerate(rows)
)
svg = f'''<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">
<rect width="100%" height="100%" fill="white"/><text x="70" y="25" font-size="15">Ridge response (left) and logistic probability (right)</text>
<line x1="{left}" y1="{bottom}" x2="{left + 290}" y2="{bottom}" stroke="black"/><line x1="{left}" y1="{probability_top}" x2="{left}" y2="{bottom}" stroke="black"/>{response_ticks}{feature_ticks}<polyline points="{line}" fill="none" stroke="#111827" stroke-width="2"/>{circles}
<text x="{left + 145}" y="{bottom + 40}" text-anchor="middle" font-size="12">synthetic feature</text><text x="18" y="205" font-size="12" transform="rotate(-90 18 205)">constructed response</text>
<circle cx="80" cy="55" r="5" fill="#2563eb"/><text x="90" y="59" font-size="11">train</text><circle cx="145" cy="55" r="5" fill="#d97706"/><text x="155" y="59" font-size="11">validation</text><circle cx="245" cy="55" r="5" fill="#dc2626"/><text x="255" y="59" font-size="11">test</text><line x1="305" y1="55" x2="325" y2="55" stroke="#111827" stroke-width="2"/><text x="332" y="59" font-size="11">selected ridge</text>
<line x1="{probability_left}" y1="{bottom}" x2="760" y2="{bottom}" stroke="black"/><line x1="{probability_left}" y1="{probability_top}" x2="{probability_left}" y2="{bottom}" stroke="black"/>{probability_ticks}<line x1="{probability_left}" y1="{probability_to_px(0.5):.1f}" x2="760" y2="{probability_to_px(0.5):.1f}" stroke="#6b7280" stroke-dasharray="4 3"/><text x="615" y="65" text-anchor="middle" font-size="11">decision threshold = 0.5</text>{bars}{entity_labels}
<text x="615" y="{bottom + 72}" text-anchor="middle" font-size="12">constructed entity ID</text><text x="428" y="205" font-size="12" transform="rotate(-90 428 205)">hit probability (0 to 1)</text></svg>'''
svg_path = OUTPUT_DIR / "fitted_models.svg"
with svg_path.open("x", encoding="utf-8") as handle:
    handle.write(svg)
display(SVG(svg))

## 10. 相对于基线阅读构造的预测失败

较低的 MAE 和 log loss 更好。在这些特意矛盾的留出标签上，所选 ridge 的 MAE 比训练均值基线
更差。Logistic 与 prevalence 的阈值化 accuracy 相同，但 logistic 对阴性测试实体给出极高置信度，
因而 log loss 更差。这些是观察到的 toy 输出，不是对真实分子预测的证据。

决策桩的测试 MAE 远低于线性模型，也低于常数。**在把它读成"胜利"之前，先读清楚原因。**
测试特征位于全部训练特征之外，而构造的响应在那里发生了转折。线性模型必须外推，
于是在上升的训练记录上拟合出的斜率把它带得远超真实值。决策桩则**根本无法外推**：
在训练范围之外它只会重复一个叶常数，而这里这种"拒绝外推"恰好落在留出响应附近。

这是关于**假设与数据规模**的论证，不是"树方法更强"的论证。两个测试实体不足以确定哪个族更好；
若决策桩落在阈值的另一侧，它同样会显得很有说服力，只是方向相反。

In [ ]:
ridge_mae = student_report["test_metrics"]["ridge"]["mae"]
stump_mae = student_report["test_metrics"]["stump"]["mae"]
mean_mae = student_report["test_metrics"]["train_mean"]["mae"]
logistic_loss = student_report["test_metrics"]["logistic"]["log_loss"]
prevalence_loss = student_report["test_metrics"]["train_prevalence"]["log_loss"]
threshold, left, right = student_report["models"]["stump"]
display(HTML(
    f"<p><b>Observed constructed comparison:</b> ridge MAE = {ridge_mae:.3f} "
    f"versus stump MAE = {stump_mae:.3f} versus training-mean MAE = {mean_mae:.3f}; "
    f"logistic log loss = {logistic_loss:.3f} versus prevalence log loss = "
    f"{prevalence_loss:.3f}.</p>"
    f"<p>The selected stump splits at {threshold} and predicts {left} below it and "
    f"{right} at or above it, so every test feature receives {right} no matter how "
    f"far outside the training range it lies. That is the assumption, not a fit.</p>"
))

## 11. 保存你观察到的合成结果

独占创建模式（`"x"`）会拒绝已有目标。第 1 步创建了全新父目录，因此从头重跑会产生单独的
报告文件夹。该 JSON 只记录这个构造练习。

In [ ]:
report_path = OUTPUT_DIR / "student_report.json"
with report_path.open("x", encoding="utf-8") as handle:
    json.dump(student_report, handle, indent=2, allow_nan=False)
    handle.write("\n")
print("Saved:", report_path)

## 12. 独立练习，再看答案

1. 给两个**测试** response 都加 100。哪些候选、选择和预测必须保持不变？哪些输出会改变？
2. 将两个验证 response 都替换为 6。哪些已拟合候选不变？选择应偏向哪个回归 alpha？
3. 复制一个实体 ID。拟合是否应开始？
4. 为什么阈值化 accuracy 可以相同，而 logistic 与 prevalence 的 log loss 却显著不同？
5. 决策桩在这些测试记录上同时胜过线性模型与常数。请指出**两个族的哪一个性质**造成了这个
   差异，并说明要让线性族反过来取胜，需要改变的是**记录**的什么（而不是模型的什么）。
6. 从 50 步到 200 步多花 900 次梯度评估，并改变了决策一致性；从 200 步到 500 步多花
   1800 次，却没有改变它。这个模式应当改变你**如何报告结果**的哪一点？以及为什么剩下的
   概率跨度仍然不是不确定性估计？

**答案。** (1) 候选参数、选择和预测不变；残差与回归测试误差会改变。(2) 候选不变；alpha 10
最接近 6。(3) 输入检查会在拟合前拒绝重复 ID。(4) Accuracy 丢弃置信度；高置信度错误概率会受到
log loss 的重罚。(5) 线性族会外推而决策桩不会：在训练范围之外，决策桩只重复一个叶常数，
而拟合出的斜率会继续上升。构造的响应恰在训练特征之外发生转折，因此"拒绝外推"反而更接近。
把测试特征放回训练范围之内，或让响应在其中继续上升，排序就会反转。两个族是在同一套程序下
选出的，所以差异来自**假设**而不是调参。这些 toy 结果不支持关于未见分子或生物学的结论，
(6) 更便宜的预算已经把决策定住了，因此多花的算力买到的是**没有人据以行动**的一致性。
应当报告"决策不再变化的那个预算"并把它写进 recipe，而不是把最大预算当成必需值引用。
剩下的跨度是三次搜索在**完全相同数据**上的距离，它可以缩到零，而那六条记录依然无法确定
答案。对后者的估计只能来自数据，不能来自优化器。
两个测试实体也不足以给模型族排序。